In [ ]:
import pandas as pd

df = pd.read_csv("/home/565/pv3484/aus_substation_electricity/figures/nsw_yearly_relative_rank.csv")


In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [ ]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

In [ ]:
obs.head

# Load in data

In [ ]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/figures/nsw_yearly_relative_rank.csv"
)


In [ ]:
rank.head()
rank["station"].unique()
rank["holiday"].unique()


In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Making dataframe

## Define timeblocks

In [ ]:
TIME_BLOCKS = {
    "00_04": ("00:00", "04:00"),
    "04_08": ("04:00", "08:00"),
    "08_15": ("08:00", "15:00"),
    "15_20": ("15:00", "20:00"),
    "20_24": ("20:00", "23:59:59")
}


## Temperature Extraction

In [ ]:
def compute_holiday_temp_24h(obs, holiday_lib, holiday_name, years):
    obs = obs.copy()
    obs.index = pd.to_datetime(obs.index)

    rows = []

    for year in years:
        holiday_date = holiday_lib[holiday_name](year)

        # Slice by date only — robust and safe
        try:
            day_slice = obs.loc[str(holiday_date.date())]
        except KeyError:
            continue

        rows.append({
            "year": year,
            "holiday": holiday_name,
            "temp_mean_24h": day_slice["t2m"].mean()
        })

    return pd.DataFrame(rows)


## Merge temp and relative rank

In [ ]:
temp_24h = compute_holiday_temp_24h(
    obs,
    HOLIDAYS_VIC,
    "ANZAC Day",
    rank["year"].unique()
)

df = rank.merge(temp_24h, on=["year", "holiday"], how="left")


In [ ]:
#df.head()
df.columns


# Plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_rank_vs_temp24(df, station, holiday):
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    # one row per year
    sub = sub.drop_duplicates(subset=["year"])

    years = sub["year"].unique()
    cmap = matplotlib.colormaps.get_cmap("hsv").resampled(len(years))

    blocks = {
        "00_04_mean": "12am–4am",
        "04_08_mean": "4am–8am",
        "08_15_mean": "8am–3pm",
        "15_20_mean": "3pm–8pm",
        "20_24_mean": "8pm–12am"
    }

    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(blocks),
        figsize=(5 * len(blocks), 5),
        sharey=True
    )

    handles = [
        plt.Line2D([0], [0], marker='o', linestyle='', color=cmap(i), markersize=8)
        for i in range(len(years))
    ]
    labels = [str(y) for y in years]

    for ax, (rank_col, label) in zip(axes, blocks.items()):
        for j, year in enumerate(years):
            row = sub[sub["year"] == year].iloc[0]
            ax.scatter(
                row["temp_mean_24h"],
                row[rank_col],
                color=cmap(j),
                s=60
            )

        ax.set_title(label)
        ax.set_xlabel(f"Mean Daily Temperature on {holiday} (°C)")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Mean Relative Rank")

    fig.legend(
        handles,
        labels,
        title="Year",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.05),
        ncol=len(labels),
        frameon=False,
        fontsize=14,
        title_fontsize=14
    )

    fig.suptitle(f"{holiday} — {station} — Mean Rank vs 24h Holiday Temperature", fontsize=16)
    fig.tight_layout()
    plt.show()


In [ ]:
plot_rank_vs_temp24(df, "MOSMA", "ANZAC Day")
